## Libraries

In [ ]:
import numpy as np
import pandas as pd
import random
import math

from tqdm import tqdm
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

## Config

In [ ]:
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")
TEST_END_DATE    = pd.Timestamp("2024-03-31")  # optional cap; set None to keep all

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Put your best tuned params here ----
best_params = {
    "WINDOW": 18,
    "HIDDEN_DIM": 128,
    "DROPOUT": 0.2,
    "LR": 5e-4,
    "WEIGHT_DECAY": 1e-4,
    "GATE_INIT": 0.85,
    "K_DIST": 8,
    "SIGMA_KM": 60.0,   # None for binary weights, or float for exp(-d/sigma)
    "K_CORR": 8,
    "HUBER_BETA": 1.0,
}

# Training settings
BATCH_SIZE    = 32
MAX_EPOCHS    = 200
PATIENCE      = 15
MAX_GRAD_NORM = 5.0

# For early stopping inside TRAIN period, hold out last VAL months of TRAIN
EARLYSTOP_VAL_MONTHS = 18

# Uncertainty settings
MC_SAMPLES = 50          # MC dropout passes on TEST
Z_975 = 1.959963984540054  # 97.5% quantile for 95% interval


# ----------------------------
# Feature lists (must match your data)
# ----------------------------
continuous_cols = [
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4",
]

categorical_cols = [
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

base_feature_cols = continuous_cols + categorical_cols

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



## Metrics

In [ ]:
## Metrics
def mae(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return float(100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y); yhat = np.asarray(yhat); y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    if not mask.any():
        return np.nan
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return float(same_dir.mean())

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)
    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    if not mask.any():
        return np.nan
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]
    return float(np.mean(np.abs(y_gr - yhat_gr)))

def morans_i(residuals, xs, ys, k=5, eps=1e-8, symmetric=True, row_standardize=True, permutations=0, random_state=42):
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    N = len(residuals)
    if N <= 1:
        return {"I": np.nan, "S0": np.nan, "z_score": np.nan, "p_value": np.nan}

    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=min(k + 1, N)).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh = indices[i]
        dist = distances[i]
        if neigh[0] == i:
            neigh = neigh[1:]
            dist = dist[1:]
        w = 1.0 / (dist + eps)
        W[i, neigh] = w

    if symmetric:
        W = 0.5 * (W + W.T)

    if row_standardize:
        rs = W.sum(axis=1, keepdims=True)
        W = np.where(rs > 0, W / (rs + eps), 0.0)

    S0 = W.sum()
    if S0 <= 0:
        return {"I": np.nan, "S0": float(S0), "z_score": np.nan, "p_value": np.nan}

    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps
    I_obs = (N / S0) * (num / den)

    out = {"I": float(I_obs), "S0": float(S0), "z_score": None, "p_value": None}

    if permutations > 0:
        rng = np.random.default_rng(random_state)
        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)
        mu = perm_I.mean()
        sd = perm_I.std(ddof=1) + eps
        z = (I_obs - mu) / sd
        extreme = np.sum(np.abs(perm_I - mu) >= np.abs(I_obs - mu))
        p = (extreme + 1) / (permutations + 1)
        out.update({"z_score": float(z), "p_value": float(p)})
    return out

# --- Interval / probabilistic metrics (Normal assumption) ---
def _phi(z):
    z = np.asarray(z, dtype=float)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * z * z)

def _Phi(z):
    # Normal CDF via math.erf (NumPy-safe)
    z = np.asarray(z, dtype=float)
    zz = z / np.sqrt(2.0)
    return 0.5 * (1.0 + np.array([math.erf(v) for v in zz], dtype=float))

def crps_gaussian(y, mu, sigma, eps=1e-12):
    y = np.asarray(y, dtype=float)
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    sigma = np.maximum(sigma, eps)
    z = (y - mu) / sigma
    return float(np.mean(sigma * (z * (2.0 * _Phi(z) - 1.0) + 2.0 * _phi(z) - 1.0 / np.sqrt(np.pi))))

def picp(y, lo, hi):
    y = np.asarray(y, dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    return float(np.mean((y >= lo) & (y <= hi)))

def piw(lo, hi):
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    return float(np.mean(hi - lo))

## Load data

In [ ]:
## Data prep: balanced panel + lags + forward-fill features (no bfill)
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"')
df = df.sort_values([TIME_COL, ENTITY_COL]).drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last").copy()

if TEST_END_DATE is not None:
    df = df[df[TIME_COL] <= TEST_END_DATE].copy()

dates = pd.Index(sorted(df[TIME_COL].unique()))
la_order = sorted(df[ENTITY_COL].unique())
T_total = len(dates)
N = len(la_order)

full_index = pd.MultiIndex.from_product([dates, la_order], names=[TIME_COL, ENTITY_COL])
df_panel = df.set_index([TIME_COL, ENTITY_COL]).reindex(full_index).sort_index()

# Forward-fill FEATURES only (never bfill)
df_panel[base_feature_cols] = df_panel[base_feature_cols].groupby(level=ENTITY_COL).ffill()

# Target lags (do not bfill target)
df_panel["price_lag1"]  = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
df_panel["price_lag12"] = df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)

# Forward-fill lag features only
df_panel[["price_lag1", "price_lag12"]] = df_panel[["price_lag1", "price_lag12"]].groupby(level=ENTITY_COL).ffill()

feature_cols = base_feature_cols + ["price_lag1", "price_lag12"]
F = len(feature_cols)

X_all = df_panel[feature_cols].to_numpy(dtype=np.float32).reshape(T_total, N, F)
y_all = df_panel[TARGET_COL].to_numpy(dtype=np.float32).reshape(T_total, N)

print("Panel shapes:", "X_all", X_all.shape, "y_all", y_all.shape)


## Adjacency builders

In [ ]:
def build_distance_knn_Ahat_from_panel(df_panel, dates, la_order, k_dist=8, sigma_km=None, eps=1e-8):
    first_date = dates[0]
    cent = df_panel.loc[(first_date, la_order), ["centroid_x", "centroid_y"]].to_numpy(dtype=np.float32)

    nbrs = NearestNeighbors(n_neighbors=k_dist + 1, algorithm="auto").fit(cent)
    dists_m, idx = nbrs.kneighbors(cent)

    Nloc = len(la_order)
    A = np.zeros((Nloc, Nloc), dtype=np.float32)

    for i in range(Nloc):
        for n in range(1, k_dist + 1):
            j = int(idx[i, n])
            d_km = float(dists_m[i, n] / 1000.0)
            w = 1.0 if sigma_km is None else float(np.exp(-d_km / (sigma_km + eps)))
            if w > A[i, j]:
                A[i, j] = w
            if w > A[j, i]:
                A[j, i] = w

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    return (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)

def build_corr_knn_Ahat_train_only(y_train_TN, k_corr=8, eps=1e-8):
    y = y_train_TN.copy()
    col_means = np.nanmean(y, axis=0)
    inds = np.where(np.isnan(y))
    if inds[0].size > 0:
        y[inds] = np.take(col_means, inds[1])

    corr = np.corrcoef(y.T)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 0.0)

    Nloc = corr.shape[0]
    A = np.zeros((Nloc, Nloc), dtype=np.float32)
    for i in range(Nloc):
        nbr_idx = np.argsort(-np.abs(corr[i]))[:k_corr]
        for j in nbr_idx:
            A[i, j] = 1.0
            A[j, i] = 1.0

    A = A + np.eye(Nloc, dtype=np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    return (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)


## Dataset and scalers

In [ ]:
class SpatioTemporalDataset(Dataset):
    """
    Produces one-step-ahead targets: for each time t >= window,
    returns X[t-window:t], y[t]
    """
    def __init__(self, X_TNF, y_TN, window):
        self.X = X_TNF
        self.y = y_TN
        self.window = int(window)
        self.T, self.N, self.F = X_TNF.shape
        self.indices = list(range(self.window, self.T))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]  # [window, N, F]
        y_t   = self.y[t]                  # [N]
        return torch.tensor(X_seq, dtype=torch.float32), torch.tensor(y_t, dtype=torch.float32)

class PerNodeRobustScaler:
    def __init__(self, eps: float = 1e-8):
        self.eps = eps
        self.center_ = None
        self.scale_  = None

    def fit(self, y_train_TN):
        y = y_train_TN.copy()
        med = np.nanmedian(y, axis=0)
        q1  = np.nanpercentile(y, 25, axis=0)
        q3  = np.nanpercentile(y, 75, axis=0)
        iqr = q3 - q1
        iqr = np.where(np.isfinite(iqr) & (iqr > self.eps), iqr, 1.0)
        self.center_ = med.astype(np.float32)
        self.scale_  = iqr.astype(np.float32)
        return self

    def transform(self, y_TN):
        y = y_TN.astype(np.float32, copy=True)
        out = (y - self.center_[None, :]) / (self.scale_[None, :] + self.eps)
        out[np.isnan(y)] = np.nan
        return out

    def inverse_transform(self, y_TN):
        y = y_TN.astype(np.float32, copy=True)
        out = y * (self.scale_[None, :] + self.eps) + self.center_[None, :]
        out[np.isnan(y)] = np.nan
        return out


def masked_huber_loss(y_hat, y_true, beta=1.0):
    mask = torch.isfinite(y_true)
    if mask.sum() == 0:
        return torch.tensor(0.0, device=y_hat.device)
    diff = y_hat[mask] - y_true[mask]
    abs_diff = diff.abs()
    loss = torch.where(
        abs_diff < beta,
        0.5 * (diff * diff) / beta,
        abs_diff - 0.5 * beta
    )
    return loss.mean()



## Model

In [ ]:
class GraphConv2GatedSeparate(nn.Module):
    def __init__(self, in_feats, out_feats, gate_init=0.85):
        super().__init__()
        self.linear_dist = nn.Linear(in_feats, out_feats)
        self.linear_corr = nn.Linear(in_feats, out_feats)

        gate_init = float(np.clip(gate_init, 1e-4, 1 - 1e-4))
        init_logit = np.log(gate_init / (1.0 - gate_init))
        self.gate_logit = nn.Parameter(torch.tensor(init_logit, dtype=torch.float32))

    def forward(self, X, A_dist, A_corr):
        AX_dist = torch.einsum("ij,bjf->bif", A_dist, X)
        AX_corr = torch.einsum("ij,bjf->bif", A_corr, X)
        out_dist = self.linear_dist(AX_dist)
        out_corr = self.linear_corr(AX_corr)
        g = torch.sigmoid(self.gate_logit)
        return g * out_dist + (1.0 - g) * out_corr

    def gate_value(self):
        return float(torch.sigmoid(self.gate_logit).detach().cpu().item())

class TGCNCell2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.85):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv2GatedSeparate(in_feats + hidden_dim, 2 * hidden_dim, gate_init=gate_init)
        self.gc_h  = GraphConv2GatedSeparate(in_feats + hidden_dim, hidden_dim,     gate_init=gate_init)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_dist, A_corr):
        if H_prev is None:
            H_prev = torch.zeros(X_t.size(0), X_t.size(1), self.hidden_dim, device=X_t.device)
        XH = torch.cat([X_t, H_prev], dim=-1)
        ZR = torch.sigmoid(self.gc_zr(XH, A_dist, A_corr))
        Z, R = torch.chunk(ZR, 2, dim=-1)
        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_dist, A_corr))
        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN2(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0, gate_init=0.85):
        super().__init__()
        self.cell = TGCNCell2(in_feats, hidden_dim, dropout=dropout, gate_init=gate_init)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_dist, A_corr):
        H = None
        for t in range(X_seq.size(1)):
            H = self.cell(X_seq[:, t], H, A_dist, A_corr)
        return self.out(H).squeeze(-1)  # [B, N]


## Indices

In [ ]:
date_to_idx = {d: i for i, d in enumerate(dates)}

train_mask_dates = (dates >= TRAIN_START_DATE) & (dates <= TRAIN_END_DATE)
test_mask_dates  = (dates >= TEST_START_DATE)
train_idxs = np.where(train_mask_dates)[0]
test_idxs  = np.where(test_mask_dates)[0]

if len(train_idxs) == 0 or len(test_idxs) == 0:
    raise ValueError("Empty train/test split. Check date ranges in Config.")

train_start_idx = int(train_idxs[0])
train_end_idx_excl = int(train_idxs[-1]) + 1
test_start_idx = int(test_idxs[0])
test_end_idx_excl = int(test_idxs[-1]) + 1

# Split TRAIN into train_fit + train_val for early stopping
val_len = min(EARLYSTOP_VAL_MONTHS, max(1, len(train_idxs) // 5))
train_val_start = train_end_idx_excl - val_len
train_fit_start = train_start_idx
train_fit_end   = train_val_start
train_val_end   = train_end_idx_excl

print(f"Train fit: {dates[train_fit_start].date()} → {dates[train_fit_end-1].date()} ({train_fit_end-train_fit_start} months)")
print(f"Train val: {dates[train_val_start].date()} → {dates[train_val_end-1].date()} ({train_val_end-train_val_start} months)")
print(f"Test     : {dates[test_start_idx].date()} → {dates[test_end_idx_excl-1].date()} ({test_end_idx_excl-test_start_idx} months)")


## Build adjacency

In [ ]:
## Build adjacency (distance is static; corr is train-only)
A_hat_dist_np = build_distance_knn_Ahat_from_panel(
    df_panel=df_panel,
    dates=dates,
    la_order=la_order,
    k_dist=int(best_params["K_DIST"]),
    sigma_km=best_params["SIGMA_KM"],
)
A_dist = torch.tensor(A_hat_dist_np, dtype=torch.float32, device=DEVICE)

y_train_for_corr = y_all[train_start_idx:train_end_idx_excl]  # full TRAIN
A_hat_corr_np = build_corr_knn_Ahat_train_only(y_train_for_corr, k_corr=int(best_params["K_CORR"]))
A_corr = torch.tensor(A_hat_corr_np, dtype=torch.float32, device=DEVICE)



## Train

In [ ]:
WINDOW = int(best_params["WINDOW"])

X_train_full = X_all[train_start_idx:train_end_idx_excl]
y_train_full = y_all[train_start_idx:train_end_idx_excl]

X_fit = X_all[train_fit_start:train_fit_end]
y_fit = y_all[train_fit_start:train_fit_end]

X_val = X_all[train_val_start:train_val_end]
y_val = y_all[train_val_start:train_val_end]

X_test = X_all[test_start_idx:test_end_idx_excl]
y_test = y_all[test_start_idx:test_end_idx_excl]

# Feature NaN fill using TRAIN feature means (features only)
def fill_X_with_train_means(X_ref_TNF, X_TNF):
    if not np.isnan(X_TNF).any():
        return X_TNF
    means = np.nanmean(X_ref_TNF.reshape(-1, F), axis=0)
    return np.where(np.isnan(X_TNF), means[None, None, :], X_TNF)

X_fit = fill_X_with_train_means(X_train_full, X_fit)
X_val = fill_X_with_train_means(X_train_full, X_val)
X_train_full = fill_X_with_train_means(X_train_full, X_train_full)
X_test = fill_X_with_train_means(X_train_full, X_test)

# X scaling (train-only, using full TRAIN for stability)
x_scaler = StandardScaler()
X_train_full_scaled = x_scaler.fit_transform(X_train_full.reshape(-1, F)).reshape(X_train_full.shape).astype(np.float32)
X_fit_scaled = x_scaler.transform(X_fit.reshape(-1, F)).reshape(X_fit.shape).astype(np.float32)
X_val_scaled = x_scaler.transform(X_val.reshape(-1, F)).reshape(X_val.shape).astype(np.float32)
X_test_scaled = x_scaler.transform(X_test.reshape(-1, F)).reshape(X_test.shape).astype(np.float32)

# y scaling: per-node robust scaler fit on TRAIN only
y_scaler = PerNodeRobustScaler().fit(y_train_full)
y_train_full_scaled = y_scaler.transform(y_train_full).astype(np.float32)
y_fit_scaled = y_scaler.transform(y_fit).astype(np.float32)
y_val_scaled = y_scaler.transform(y_val).astype(np.float32)
y_test_scaled = y_scaler.transform(y_test).astype(np.float32)

# Build VAL context (needs WINDOW months of history from fit tail)
if X_fit_scaled.shape[0] < WINDOW + 1:
    raise ValueError("Not enough train_fit months for the chosen WINDOW. Reduce WINDOW or adjust split.")
X_val_context = np.concatenate([X_fit_scaled[-WINDOW:], X_val_scaled], axis=0).astype(np.float32)
y_val_context = np.concatenate([y_fit_scaled[-WINDOW:], y_val_scaled], axis=0).astype(np.float32)

train_ds = SpatioTemporalDataset(X_fit_scaled, y_fit_scaled, window=WINDOW)
val_ds   = SpatioTemporalDataset(X_val_context, y_val_context, window=WINDOW)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


## Train with early stopping
model = TGCN2(
    in_feats=F,
    hidden_dim=int(best_params["HIDDEN_DIM"]),
    dropout=float(best_params["DROPOUT"]),
    gate_init=float(best_params["GATE_INIT"]),
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(best_params["LR"]),
    weight_decay=float(best_params["WEIGHT_DECAY"]),
)

best_val = np.inf
best_epoch = -1
best_state = None
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    tr_losses = []
    for X_seq, y_t in train_loader:
        X_seq = X_seq.to(DEVICE)
        y_t   = y_t.to(DEVICE)

        optimizer.zero_grad()
        y_hat = model(X_seq, A_dist, A_corr)
        loss = masked_huber_loss(y_hat, y_t, beta=float(best_params["HUBER_BETA"]))

        if not torch.isfinite(loss) or (loss.item() == 0.0 and (not torch.isfinite(y_t).any())):
            continue

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        tr_losses.append(loss.item())

    if len(tr_losses) == 0:
        raise RuntimeError("No valid training batches. Check missingness / window / scaling.")

    model.eval()
    va_losses = []
    with torch.no_grad():
        for X_seq, y_t in val_loader:
            X_seq = X_seq.to(DEVICE)
            y_t   = y_t.to(DEVICE)
            y_hat = model(X_seq, A_dist, A_corr)
            vloss = masked_huber_loss(y_hat, y_t, beta=float(best_params["HUBER_BETA"]))
            if torch.isfinite(vloss) and torch.isfinite(y_t).any():
                va_losses.append(vloss.item())

    if len(va_losses) == 0:
        raise RuntimeError("No valid validation batches. Check missingness / window / scaling.")

    val_loss = float(np.mean(va_losses))
    g_zr = model.cell.gc_zr.gate_value()
    g_h  = model.cell.gc_h.gate_value()
    print(f"Epoch {epoch:03d} | trainHuber={np.mean(tr_losses):.4f} | valHuber={val_loss:.4f} | gates(zr={g_zr:.3f}, h={g_h:.3f})")

    if val_loss + 1e-6 < best_val:
        best_val = val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, best val {best_val:.4f})")
            break

if best_state is None:
    raise RuntimeError("Training failed: no best_state captured.")
model.load_state_dict(best_state)


## Helpers: deterministic + MC-dropout prediction on a loader
def predict_loader(model, loader, A_dist, A_corr, mc_samples=0):
    """
    Returns:
      y_true_scaled: [M, N]
      y_pred_mean_scaled: [M, N]
      y_pred_std_scaled: [M, N]  (0 if mc_samples<=1)
    """
    # collect truths once
    model.eval()
    y_true_list = []
    with torch.no_grad():
        for X_seq, y_t in loader:
            y_true_list.append(y_t.numpy())
    y_true_scaled = np.concatenate(y_true_list, axis=0).astype(np.float32)

    if mc_samples is None or mc_samples <= 1:
        model.eval()
        preds = []
        with torch.no_grad():
            for X_seq, _ in loader:
                X_seq = X_seq.to(DEVICE)
                y_hat = model(X_seq, A_dist, A_corr).detach().cpu().numpy()
                preds.append(y_hat)
        y_pred = np.concatenate(preds, axis=0).astype(np.float32)
        y_std  = np.zeros_like(y_pred, dtype=np.float32)
        return y_true_scaled, y_pred, y_std

    # MC dropout: enable dropout by setting train() but keep no_grad
    samples = []
    for s in range(int(mc_samples)):
        model.train()
        preds_s = []
        with torch.no_grad():
            for X_seq, _ in loader:
                X_seq = X_seq.to(DEVICE)
                y_hat = model(X_seq, A_dist, A_corr).detach().cpu().numpy()
                preds_s.append(y_hat)
        samples.append(np.concatenate(preds_s, axis=0).astype(np.float32))

    S = np.stack(samples, axis=0)  # [S, M, N]
    return y_true_scaled, S.mean(axis=0), S.std(axis=0, ddof=1)


## Build TEST context loader (needs WINDOW months of history from end of TRAIN)
X_test_context = np.concatenate([X_train_full_scaled[-WINDOW:], X_test_scaled], axis=0).astype(np.float32)
y_test_context = np.concatenate([y_train_full_scaled[-WINDOW:], y_test_scaled], axis=0).astype(np.float32)

test_ds = SpatioTemporalDataset(X_test_context, y_test_context, window=WINDOW)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Also build TRAIN loader for sigma estimation (on full TRAIN)
train_full_ds = SpatioTemporalDataset(X_train_full_scaled, y_train_full_scaled, window=WINDOW)
train_full_loader = DataLoader(train_full_ds, batch_size=BATCH_SIZE, shuffle=False)


## Predict: TRAIN (deterministic) for Sigma_hat_train, TEST (MC dropout) for intervals
ytr_true_sc, ytr_pred_sc, _ = predict_loader(model, train_full_loader, A_dist, A_corr, mc_samples=0)

yte_true_sc, yte_pred_mean_sc, yte_pred_std_sc = predict_loader(model, test_loader, A_dist, A_corr, mc_samples=MC_SAMPLES)

# Inverse transform to original units
ytr_true = y_scaler.inverse_transform(ytr_true_sc)
ytr_pred = y_scaler.inverse_transform(ytr_pred_sc)

yte_true = y_scaler.inverse_transform(yte_true_sc)
yte_pred_mean = y_scaler.inverse_transform(yte_pred_mean_sc)

# For std: robust scaler is affine per node, so std transforms by scale_
# y = center + scale * z  => std_y = |scale| * std_z
node_scale = y_scaler.scale_.astype(np.float32)  # [N]
yte_pred_std = yte_pred_std_sc * node_scale[None, :]

# Sigma_hat_train: aleatoric noise estimated from TRAIN residuals (orig units)
mask_tr = np.isfinite(ytr_true) & np.isfinite(ytr_pred)
train_resid = (ytr_true - ytr_pred)[mask_tr]
Sigma_hat_train = float(np.sqrt(np.mean(train_resid ** 2))) if train_resid.size else np.nan

# Total predictive sd: combine epistemic (MC dropout) + aleatoric (Sigma_hat_train)
yte_pred_sd_total = np.sqrt(np.maximum(yte_pred_std ** 2 + (Sigma_hat_train ** 2), 0.0)).astype(np.float32)

# Flatten TEST to a long dataframe aligned to dates
test_dates = dates[test_start_idx:test_end_idx_excl]
# test_ds yields one row per time step in context starting at t=WINDOW,
# which corresponds exactly to each test date (because we prepended WINDOW train months).
assert len(test_ds) == len(test_dates), "Date alignment mismatch for TEST."

# Build long-form table
rows = []
for t_i, d in enumerate(test_dates):
    y_true_t = yte_true[t_i]
    y_pred_t = yte_pred_mean[t_i]
    y_sd_t   = yte_pred_sd_total[t_i]
    for j, la in enumerate(la_order):
        yt = y_true_t[j]
        if not np.isfinite(yt):
            continue
        mu = float(y_pred_t[j])
        sd = float(y_sd_t[j])
        rows.append({
            TIME_COL: d,
            ENTITY_COL: la,
            TARGET_COL: float(yt),
            "y_pred": mu,
            "y_pred_sd": sd,
            "pi95_lo": mu - Z_975 * sd,
            "pi95_hi": mu + Z_975 * sd,
        })

df_test_long = pd.DataFrame(rows)
df_test_long["resid"] = df_test_long[TARGET_COL] - df_test_long["y_pred"]


## Results: point metrics
y_test_vec = df_test_long[TARGET_COL].values
y_pred_vec = df_test_long["y_pred"].values

# For MASE denominator use TRAIN observed values (flattened)
y_train_vec = y_train_full.reshape(-1)
y_train_vec = y_train_vec[np.isfinite(y_train_vec)]


## Evaluation

In [ ]:
global_mae   = mae(y_test_vec, y_pred_vec)
global_rmse  = rmse(y_test_vec, y_pred_vec)
global_smape = smape(y_test_vec, y_pred_vec)
global_mase  = mase(y_test_vec, y_pred_vec, y_train_vec, m=12)

print("=== Global accuracy (T-GCN) ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# Interval / probabilistic metrics
PICP_95 = picp(y_test_vec, df_test_long["pi95_lo"].values, df_test_long["pi95_hi"].values)
PIW_95  = piw(df_test_long["pi95_lo"].values, df_test_long["pi95_hi"].values)
CRPS    = crps_gaussian(y_test_vec, y_pred_vec, df_test_long["y_pred_sd"].values)

print("\n=== Uncertainty / interval metrics ===")
print(f"Sigma_hat_train (RMSE on TRAIN residuals): {Sigma_hat_train:,.4f}")
print(f"PICP_95  : {PICP_95:,.3f}")
print(f"PIW_95   : {PIW_95:,.3f}")
print(f"CRPS     : {CRPS:,.4f}")

# Across-LA consistency
la_mae = df_test_long.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))
median_mae = float(la_mae.median())
p75_mae    = float(la_mae.quantile(0.75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# Spatio-temporal diagnostics
# Moran's I on mean residual per LA over test period
la_resid_mean = df_test_long.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_panel
    .reset_index()
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on monthly mean residuals
monthly_resid = df_test_long.groupby(TIME_COL)["resid"].mean().sort_index()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])
print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# Direction & growth
dir_acc = directional_accuracy(df_test_long, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test_long, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


## Output

In [ ]:
output_path = "../../results/tgcn_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "TGCN_2Channel_Gated",
    "best_params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,
    # requested
    "PICP_95": PICP_95,
    "PIW_95": PIW_95,
    "CRPS": CRPS,
    "Sigma_hat_train": Sigma_hat_train,
    # helpful context
    "MC_SAMPLES": int(MC_SAMPLES),
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test_long[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "y_pred_sd", "pi95_lo", "pi95_hi", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")